In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 4.7 The Relativistic Lagrangian and Motion in Fields

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume IV — Special Relativity",
    number="4.7",
    title="The Relativistic Lagrangian and Motion in Fields",
    blurb="Mechanics and electrodynamics, made relativistic and reunited: the "
    "free particle extremizes its proper time, and a charge in "
    "electromagnetic fields moves under the covariant Lorentz force built "
    "from the field tensor of §3.12.",
    difficulty="advanced",
    estimate="130–160 min",
)

## Notebook overview

This notebook is where two long threads of the course tie together. Volume II made
mechanics variational — motion extremizes an action — and Volume III built
electrodynamics, culminating in the field tensor $F^{\mu\nu}$ of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) that fused $\mathbf
E$ and $\mathbf B$ into one object. Here both become relativistic and meet. The free
relativistic particle turns out to extremize its **proper time**, so the fact of [§4.4](paradoxes.ipynb) that the
inertial worldline is the longest becomes a genuine variational principle; and a charge in
a field moves under the **covariant Lorentz force** $dp^\mu/d\tau=qF^{\mu\nu}u_\nu$, the
tensor of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) now driving dynamics. This is the explicit Volume III $\leftrightarrow$
Volume IV handshake.

From the single Lagrangian $L=-mc^2\sqrt{1-v^2/c^2}$ we recover the canonical momentum
$\gamma m\mathbf v$ and energy $\gamma mc^2$ of [§4.5](four-momentum-energy.ipynb) — now from a variational principle
rather than a postulate — and, with Noether's theorem from [§2.2](../02-classical-mechanics/noether.ipynb), the conservation of
four-momentum. Adding the electromagnetic coupling gives the equation of motion, whose
covariant form we verify reproduces the familiar $q(\mathbf E+\mathbf v\times\mathbf B)$ by
contracting the field tensor of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) with the four-velocity. We then put a real particle in
real fields and integrate its trajectory: the **relativistic cyclotron**, whose frequency
$\omega=qB/(\gamma m)$ falls with energy (the flaw that forced the invention of the
synchrotron), and the $\mathbf E\times\mathbf B$ **drift** of crossed fields, a building
block of plasma physics.

We reuse the four-vector and `np.einsum` machinery of [§4.3](spacetime-minkowski.ipynb) and [§4.5](four-momentum-energy.ipynb) and the field tensor of
[§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) by cross-reference, not rederivation. Trajectories are integrated with
`scipy.integrate.solve_ivp` (DOP853), with the relativistic subtlety handled by carrying the
**momentum** as the state variable and recovering the velocity from it, so the
$\gamma$-dependence never needs inverting by hand. One trajectory, the cyclotron orbit, is
genuine motion and is animated; the rest are clean stills.

Everything is in **SI units**, with $c=1/\sqrt{\mu_0\varepsilon_0}=2.998\times10^8\,$m/s and
stated charges, masses, fields, and initial conditions.

> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the canonical momentum $\gamma mv$; the covariant force equal to
> $q(\mathbf E+\mathbf v\times\mathbf B)$; the conserved speed and the cyclotron frequency
> $qB/(\gamma m)$; the $1/\gamma$ falloff measured from integrated orbits; the $\mathbf E
> \times\mathbf B$ drift $E/B$ in direction and magnitude; the momentum growing as exactly
> $qEt$ while $v\to c$; four-momentum conserved for the free particle.
> A ✓ is strong evidence; a ✗ is a prompt to *locate the discrepancy*, not a verdict.
>
> **Scope.** The relativistic Lagrangian and charged-particle motion; curved spacetime is
> the capstone [§4.8](gr-capstone.ipynb). See Landau & Lifshitz, *The Classical Theory of Fields*; Nolting,
> *Theoretical Physics 4* {cite}`nolting4`; Jackson, *Classical Electrodynamics*; and
> [§2.1](../02-classical-mechanics/lagrangian-sympy.ipynb)/[§2.2](../02-classical-mechanics/noether.ipynb) (Lagrangian mechanics, Noether), [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) (the field tensor), [§4.5](four-momentum-energy.ipynb)
> (four-momentum).

## Theory in brief

### The relativistic action: extremize proper time

The free relativistic particle has the action

```{math}
:label: eq-rel-action
S=-mc^2\int d\tau=-mc\int\sqrt{-dx_\mu dx^\mu}, \qquad L=-mc^2\sqrt{1-v^2/c^2},
```

proportional to the **proper time** along the worldline. Extremizing it with the
Euler–Lagrange machinery of [§2.1](../02-classical-mechanics/lagrangian-sympy.ipynb) gives straight-line motion in flat spacetime, so the
result of [§4.4](paradoxes.ipynb) — the inertial worldline maximizes proper time — is now a variational principle.

### Canonical momentum and energy recovered

From this $L$ the canonical momentum and energy are

```{math}
:label: eq-rel-canonical
\mathbf p=\frac{\partial L}{\partial\mathbf v}=\gamma m\mathbf v, \qquad E=\mathbf p\cdot\mathbf v-L=\gamma mc^2,
```

the four-momentum of [§4.5](four-momentum-energy.ipynb), now from a variational principle. By Noether's theorem ([§2.2](../02-classical-mechanics/noether.ipynb)),
time-translation symmetry conserves $E=\gamma mc^2$ and space-translation conserves
$\mathbf p=\gamma m\mathbf v$ — together, four-momentum conservation.

### Adding the electromagnetic interaction

A charge couples to the four-potential $A^\mu$ ([§3.6](../03-electrodynamics/magnetostatics.ipynb), [§3.8](../03-electrodynamics/maxwell-waves.ipynb), [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb)) through $q\int A_\mu
dx^\mu$, the simplest Lorentz-invariant coupling of a charge to a field (Landau &
Lifshitz, *The Classical Theory of Fields*, § 16, carries the construction out in full),
giving the full Lagrangian

```{math}
:label: eq-rel-em-action
L=-mc^2\sqrt{1-v^2/c^2}-qV+q\mathbf A\cdot\mathbf v .
```

### The covariant Lorentz force

Euler–Lagrange on that $L$ yields the equation of motion in covariant form (Jackson,
*Classical Electrodynamics*, Ch. 12, carries the variation out in full),

```{math}
:label: eq-covariant-force
\frac{dp^\mu}{d\tau}=qF^{\mu\nu}u_\nu,
```

with $F^{\mu\nu}$ the field tensor of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) and $u^\nu$ the four-velocity. Its spatial part
is the familiar Lorentz force $d\mathbf p/dt=q(\mathbf E+\mathbf v\times\mathbf B)$ (with
$\mathbf p=\gamma m\mathbf v$), and its time part is the power $q\mathbf E\cdot\mathbf v$.
The tensor built in the electrodynamics capstone now drives relativistic dynamics.

### Relativistic cyclotron motion

A charge in a uniform $\mathbf B$ moves in a circle, but the cyclotron frequency (read
off from $d\mathbf p/dt=q\,\mathbf v\times\mathbf B$ with $\mathbf p=\gamma m\mathbf v$)

```{math}
:label: eq-rel-cyclotron
\omega=\frac{qB}{\gamma m}, \qquad r=\frac{\gamma mv}{qB},
```

*decreases* with energy through the $1/\gamma$. The Newtonian $\omega=qB/m$ is the
low-energy limit. This energy-dependence is exactly why a fixed-frequency cyclotron loses
sync with the particle, and why the synchrotron was invented.

### Crossed fields and drifts

In perpendicular $\mathbf E$ and $\mathbf B$ the guiding center drifts at

```{math}
:label: eq-drift
\mathbf v_{\rm drift}=\frac{\mathbf E\times\mathbf B}{B^2},
```

of magnitude $E/B$, independent of charge and mass — a building block of plasma physics
(Jackson, *Classical Electrodynamics*, § 12.3, derives the drift in full).

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp

from ecp import draw, validate
from ecp.animate import show

from scipy.constants import mu_0 as MU0  # vacuum permeability, T·m/A
from scipy.constants import epsilon_0 as EPS0  # vacuum permittivity, F/m

C_LIGHT = 1.0 / np.sqrt(MU0 * EPS0)  # speed of light, m/s
from scipy.constants import e as Q_E  # elementary charge, C
from scipy.constants import m_p as M_P  # proton mass, kg

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT

# The Minkowski metric — the toolkit of §4.3/§4.5, reused here.
ETA = np.diag([-1.0, 1.0, 1.0, 1.0])  # η = diag(−1, 1, 1, 1), signature (−,+,+,+)


def gamma(v):
    """The Lorentz factor γ = 1/√(1 - v^2/c^2) for a speed ``v`` in m/s (eq-rel-canonical).

    Parameters
    ----------
    v : float or numpy.ndarray
        Speed, in m/s (|v| < c).

    Returns
    -------
    float or numpy.ndarray
        The Lorentz factor.
    """
    return 1.0 / np.sqrt(1.0 - (v / C_LIGHT) ** 2)


def field_tensor(E, B):
    """The contravariant field tensor F^μν from E and B (the §3.12 builder, reused).

    Packs the six field components into one antisymmetric 4×4 matrix: E in the time
    row/column (F^{0i} = E_i/c) and B in the spatial block (F^{ij} = -ε^{ijk} B_k).
    Identical to the construction of §3.12.

    Parameters
    ----------
    E : array_like
        Electric field (E_x, E_y, E_z), in V/m.
    B : array_like
        Magnetic field (B_x, B_y, B_z), in T.

    Returns
    -------
    numpy.ndarray
        The 4×4 tensor F^μν.
    """
    Ex, Ey, Ez = E
    Bx, By, Bz = B
    c = C_LIGHT
    return np.array(
        [
            [0.0, Ex / c, Ey / c, Ez / c],
            [-Ex / c, 0.0, Bz, -By],
            [-Ey / c, -Bz, 0.0, Bx],
            [-Ez / c, By, -Bx, 0.0],
        ]
    )


def velocity_from_momentum(p, m):
    """Recover the 3-velocity from the relativistic momentum, v = p c^2/E (eq-rel-canonical).

    With E = √((pc)^2 + (mc^2)^2), this inverts p = γmv without solving for γ explicitly,
    which is why the trajectory integrators carry momentum as the state variable.

    Parameters
    ----------
    p : numpy.ndarray
        Relativistic momentum (p_x, p_y, p_z), in kg·m/s.
    m : float
        Rest mass, in kg.

    Returns
    -------
    numpy.ndarray
        The 3-velocity (v_x, v_y, v_z), in m/s.
    """
    E = np.sqrt((np.linalg.norm(p) * C_LIGHT) ** 2 + (m * C_LIGHT**2) ** 2)
    return p * C_LIGHT**2 / E


def lorentz_rhs(t, state, q, m, E, B):
    """Right-hand side of the relativistic equation of motion for ``solve_ivp`` (eq-covariant-force).

    The state is (r, p); dr/dt is the velocity recovered from p (so the γ-dependence is
    never inverted by hand), and dp/dt = q(E + v×B) is the spatial covariant Lorentz force.

    Parameters
    ----------
    t : float
        Time (unused; the fields are static here).
    state : numpy.ndarray
        The six-component state (r_x, r_y, r_z, p_x, p_y, p_z).
    q, m : float
        Charge (C) and rest mass (kg).
    E, B : numpy.ndarray
        Uniform electric (V/m) and magnetic (T) fields.

    Returns
    -------
    numpy.ndarray
        The time derivative (dr/dt, dp/dt).
    """
    p = state[3:]
    v = velocity_from_momentum(p, m)
    dpdt = q * (E + np.cross(v, B))  # spatial Lorentz force, dp/dt
    return np.concatenate([v, dpdt])

## Exercise 1 — The free relativistic Lagrangian

Volume II taught that motion extremizes an action. The relativistic free particle has the
strikingly simple action $S=-mc^2\int d\tau$ {eq}`eq-rel-action`: it is, up to a constant,
the **proper time** along the worldline. Extremizing it therefore extremizes proper time,
which is precisely the statement of [§4.4](paradoxes.ipynb) that the inertial worldline is the longest — now a
variational principle rather than an observation. In a chosen frame the Lagrangian is
$L=-mc^2\sqrt{1-v^2/c^2}$, and from it the canonical momentum and energy {eq}`eq-rel-canonical`
must reproduce the four-momentum of [§4.5](four-momentum-energy.ipynb).

**This exercise (worked).** For a proton at $v=0.6c$, compute the canonical momentum
$\partial L/\partial v$ two ways — symbolically with `sympy.diff` and numerically by a
central finite difference — and confirm both equal $\gamma mv$; then form the energy
$pv-L$ and confirm it equals $\gamma mc^2$. The action is proper time, and its variational
structure hands back the dynamics of [§4.5](four-momentum-energy.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    p_symbolic,
    gamma(v0) * m * v0,
    "the relativistic Lagrangian gives canonical momentum γmv (symbolic)",
    rtol=1e-9,
)
validate.close(
    p_numerical,
    gamma(v0) * m * v0,
    "the canonical momentum by finite difference also equals γmv",
    rtol=1e-4,
)
validate.close(
    E_canonical, gamma(v0) * m * C_LIGHT**2, "the energy pv − L equals γmc²", rtol=1e-4
)

## Exercise 2 — The covariant Lorentz force

Here is the handshake. Adding the electromagnetic coupling to the action gives an equation
of motion that, written covariantly, is $dp^\mu/d\tau=qF^{\mu\nu}u_\nu$ {eq}`eq-covariant-force`,
where $F^{\mu\nu}$ is *exactly* the field tensor built in [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) and $u^\nu$ the four-velocity
of [§4.5](four-momentum-energy.ipynb). The single tensor that unified $\mathbf E$ and $\mathbf B$ now moves a charge. Its
spatial part must reproduce the Lorentz force $q(\mathbf E+\mathbf v\times\mathbf B)$ and
its time part the power $q\mathbf E\cdot\mathbf v$, which we verify by direct contraction.

**This exercise (worked).** For stated $\mathbf E$, $\mathbf B$, and a velocity $\mathbf v$,
build $F^{\mu\nu}$ with the `field_tensor` of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb), lower the four-velocity's index with the
metric (`ETA @ u`), contract $qF^{\mu\nu}u_\nu$ with `np.einsum('mn,n->m', ...)`, and confirm
the spatial part equals $q(\mathbf E+\mathbf v\times\mathbf B)$ (`np.cross`) and the time part
equals the power $q\mathbf E\cdot\mathbf v$.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    dpdt_spatial,
    lorentz_force,
    "the covariant force dp^μ/dτ=qF^μν u_ν reproduces the Lorentz force q(E+v×B)",
    rtol=1e-6,
)
validate.close(
    power_from_force,
    q * (E_field @ v_vec),
    "the time part of the covariant force is the power qE·v",
    rtol=1e-6,
)

## Exercise 3 — Relativistic cyclotron motion

Now we let a charge actually move. In a uniform magnetic field the magnetic force is always
perpendicular to the velocity, so it does no work: the speed is constant and the orbit is a
circle ({numref}`fig-rl-cyclotron`). But the circulation frequency carries a relativistic
surprise, $\omega=qB/(\gamma m)$ {eq}`eq-rel-cyclotron`, smaller than the Newtonian $qB/m$ by
$1/\gamma$ because the moving mass is harder to turn. We integrate the relativistic equation
of motion, carrying the momentum as the state so the $\gamma$-dependence is handled cleanly,
and read the frequency off the trajectory.

**This exercise (worked).** For a proton at $v_0=0.6c$ in $B=0.7\,$T, integrate
$d\mathbf p/dt=q\,\mathbf v\times\mathbf B$ with `scipy.integrate.solve_ivp` (DOP853),
recovering $\mathbf v$ from $\mathbf p$ each step. Confirm the speed is conserved (the
magnetic force does no work) and that the measured angular frequency equals $qB/(\gamma m)$.
The animation traces the circular orbit.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    speeds,
    np.full_like(speeds, v_cyc),
    "the magnetic force does no work — the speed is conserved",
    rtol=1e-6,
)
validate.close(
    omega_measured,
    omega_expected,
    "the relativistic cyclotron frequency is qB/(γm)",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Why the cyclotron fails relativistically

The energy-dependence of the cyclotron frequency is not a curiosity but a piece of
accelerator history. Because $\omega=qB/(\gamma m)$ {eq}`eq-rel-cyclotron` falls as the
particle gains energy, a cyclotron driven at a *fixed* frequency — tuned to the
non-relativistic $qB/m$ — gradually slips out of step with the particle it is meant to push,
and the acceleration stalls ({numref}`fig-rl-omega`). The cure was to vary the driving
frequency (the synchrocyclotron) or the field (the synchrotron) to track the falling
$\omega$. The relativistic correction sets a hard ceiling on the plain cyclotron.

**This exercise (worked).** *Measure* the falloff rather than restate the formula: for
$\beta=0.1,\,0.5,\,0.9$, integrate the cyclotron orbit with `scipy.integrate.solve_ivp`
(DOP853) exactly as in Exercise 3, extract each orbit's angular frequency from the unwrapped
velocity angle with `numpy.polyfit`, and confirm that the product $\gamma\,\omega_{\rm
measured}$ is the same Newtonian constant $qB/m$ at every speed — the measured frequency
falls as $1/\gamma$. The figure shows the falloff against the Newtonian constant.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    gamma_omega,
    np.full(3, omega_newton),
    "the measured cyclotron frequency falls as 1/γ (γ·ω equals the Newtonian qB/m)",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Crossed fields and the E×B drift

When $\mathbf E$ and $\mathbf B$ are perpendicular, something elegant happens: superposed on
the circular gyration, the guiding center *drifts* sideways at a constant velocity
$\mathbf v_{\rm drift}=(\mathbf E\times\mathbf B)/B^2$ {eq}`eq-drift`, of magnitude $E/B$ and —
remarkably — independent of the particle's charge and mass. Electrons and protons drift
together, which is why the $\mathbf E\times\mathbf B$ drift is foundational in plasma physics.
The trajectory is a cycloid: a fast little circle carried along by the steady drift
({numref}`fig-rl-drift`).

**This exercise (worked).** For $\mathbf E=10^6\,$V/m along $y$ and $\mathbf B=0.1\,$T along
$z$ (so $E/B=10^7\,$m/s, well below $c$, and the drift runs along $x$), integrate a proton
starting from rest with `solve_ivp` (DOP853), and confirm its mean velocity — the drift —
equals $(\mathbf E\times\mathbf B)/B^2$ in both direction and magnitude $E/B$.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    drift_measured,
    drift_expected,
    "the measured drift equals (E×B)/B² in direction and magnitude E/B",
    rtol=1e-2,
    atol=1e-2 * np.linalg.norm(drift_expected),
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — A relativistic charge accelerated by E

A constant electric field exerts a constant force, and in Newtonian mechanics that means a
constant acceleration and a speed growing without bound. Relativity forbids the latter: the
force still adds momentum at a constant rate, so $p=qEt$ grows linearly forever, but the
speed recovered from that momentum *saturates* at $c$ and never reaches it
({numref}`fig-rl-accel`). This is the dynamical speed limit of [§4.5](four-momentum-energy.ipynb) seen as a trajectory — the
energy cost of the last increment of speed diverges, even though nothing stops us pushing.

**This exercise (student).** Integrate a proton starting from rest in $\mathbf E=10^7\,$V/m
with `solve_ivp` (DOP853), confirm the integrated momentum matches the exact constant-force
law $p=qEt$ at every step, and confirm with a `numpy` comparison that the final speed is very
close to but below $c$ while the momentum has grown far beyond $mc$ and is still climbing
linearly.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    p_mag,
    q * np.linalg.norm(E_accel) * t_eval6,
    "a constant force adds momentum at a constant rate — the integrated p equals qEt exactly",
    rtol=1e-8,
)
validate.check(
    asymptotes,
    "a constant force accelerates a charge toward c but never to it — p grows unbounded, v saturates",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Noether for the relativistic particle

Volume II's Noether theorem ([§2.2](../02-classical-mechanics/noether.ipynb)) ties each continuous symmetry of the action to a conserved
quantity, and the relativistic free particle is no exception. Its Lagrangian $L=-mc^2\sqrt{1
-v^2/c^2}$ has no explicit dependence on time or position, so time-translation symmetry
conserves the energy $\gamma mc^2$ and space-translation symmetry conserves the momentum
$\gamma m\mathbf v$ {eq}`eq-rel-canonical` — together, the conservation of four-momentum that
[§4.5](four-momentum-energy.ipynb) asserted. We confirm it directly along an integrated free trajectory.

**This exercise (student).** Integrate a free proton (no fields) at $\mathbf v=(0.5,0.3,0)c$
with `solve_ivp` (DOP853), and confirm with `numpy` that the energy $\gamma mc^2$ and each
component of the momentum $\gamma m\mathbf v$ stay constant along the worldline — Noether's
theorem made numerical.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    energies,
    np.full_like(energies, energies[0]),
    "Noether: time-translation symmetry conserves the energy γmc²",
    rtol=1e-8,
)
validate.close(
    mom7[:, 0],
    np.full_like(mom7[:, 0], p_free[0]),
    "Noether: space-translation symmetry conserves the momentum γmv",
    rtol=1e-8,
)

## Exercise 8 — One mechanics, one electrodynamics, made relativistic

Look at what has converged in this notebook. Volume II's variational mechanics found its
relativistic form in an action that is nothing but the proper time, handing back the
four-momentum of [§4.5](four-momentum-energy.ipynb) and, through Noether, its conservation. Volume III's electrodynamics
found its dynamical form in the covariant Lorentz force $dp^\mu/d\tau=qF^{\mu\nu}u_\nu$, the
field tensor of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) now setting a charge in motion. The four-vectors of [§4.3](spacetime-minkowski.ipynb), the
four-momentum of [§4.5](four-momentum-energy.ipynb), the field tensor of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb), and the Noether theorem of [§2.2](../02-classical-mechanics/noether.ipynb) are not
separate results but one structure seen from different sides. And the central idea — that a
free particle extremizes its proper time — is the seed of the capstone: in *curved*
spacetime, extremizing proper time becomes motion along geodesics, which is gravity itself.

**This exercise.** Close with the unifying check: confirm with `np.allclose` that the same
covariant force law, contracted from the field tensor of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb), reproduces the elementary Lorentz
force in three different field configurations (pure $\mathbf E$, pure $\mathbf B$, and crossed
fields) — one equation of motion underlying every case in this notebook.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    all_match,
    "one covariant force law (from the §3.12 field tensor) underlies every field configuration",
)

## Notebook summary

- **The free relativistic Lagrangian** {eq}`eq-rel-action`, {eq}`eq-rel-canonical`:
  $L=-mc^2\sqrt{1-v^2/c^2}$ is (minus) the proper time, so extremizing the action extremizes
  proper time (the result of [§4.4](paradoxes.ipynb) as a variational principle); $\partial L/\partial v=\gamma mv$
  (symbolic and finite-difference) and $pv-L=\gamma mc^2$ recover the four-momentum of [§4.5](four-momentum-energy.ipynb).
- **The covariant Lorentz force** {eq}`eq-covariant-force`: $dp^\mu/d\tau=qF^{\mu\nu}u_\nu$,
  built from the field tensor of [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) and contracted with `np.einsum`, reproduces $q(\mathbf E+
  \mathbf v\times\mathbf B)$ in its spatial part and the power $q\mathbf E\cdot\mathbf v$ in its
  time part — the Volume III $\leftrightarrow$ IV handshake.
- **Relativistic cyclotron** {eq}`eq-rel-cyclotron`: the orbit is circular with conserved
  speed, and the frequency measured from integrated orbits at $\beta=0.1$–$0.9$ is
  $\omega=qB/(\gamma m)$, the Newtonian $qB/m$ divided by $\gamma$ — the falloff that doomed
  the fixed-frequency cyclotron and forced the synchrotron.
- **Crossed fields** {eq}`eq-drift`: the guiding center drifts at $(\mathbf E\times\mathbf B)/
  B^2$, magnitude $E/B=10^7\,$m/s, independent of charge and mass — the plasma-physics drift.
- **The dynamical speed limit and Noether**: a constant $\mathbf E$ grows $p=qEt$ without
  bound while $v\to c$ ([§4.5](four-momentum-energy.ipynb) as a trajectory); and the free particle conserves $\gamma mc^2$ and
  $\gamma m\mathbf v$ by Noether's theorem ([§2.2](../02-classical-mechanics/noether.ipynb)).

Variational mechanics (II) and electrodynamics (III), both made relativistic, meet in one
action and one covariant force.

## Outlook

- **The general-relativity capstone ([§4.8](gr-capstone.ipynb)).** Extremizing proper time in *curved* spacetime is
  geodesic motion — gravity as geometry. The free-particle action here is the flat-spacetime
  seed of it.
- **Particle accelerators.** The synchrotron that the cyclotron's $1/\gamma$ falloff demanded,
  and synchrotron radiation (a callback to [§3.10](../03-electrodynamics/radiation.ipynb)).
- **Plasma physics.** Guiding-center drifts and magnetic mirrors, built on the $\mathbf E
  \times\mathbf B$ drift (a pointer).
- **Classical field theory.** The field Lagrangian and the action for the electromagnetic
  field itself (Landau & Lifshitz) — a pointer beyond the course.
- **Cross-reference** [§2.1](../02-classical-mechanics/lagrangian-sympy.ipynb)/[§2.2](../02-classical-mechanics/noether.ipynb) (Lagrangian mechanics and Noether), [§3.12](../03-electrodynamics/relativistic-maxwell.ipynb) (the field tensor),
  and [§4.5](four-momentum-energy.ipynb) (four-momentum).

In [ ]:
from ecp.style import footer

footer()